# 12 · Clasificación con regresión logística

**Pregunta de clase.** ¿Podemos estimar si una persona terminará el curso? La regresión logística calcula una probabilidad para un resultado de dos clases. La compararemos con una regla sencilla.

## 0 · Preparar el entorno

Elige un kernel de **Python 3.11 o 3.12** con **Java 17**. En Jupyter o VS Code usa el mismo entorno que en las prácticas anteriores. En Colab, comprueba la versión con `!python --version`. Si muestra Python 3.13, en **Entorno de ejecución → Cambiar tipo de entorno de ejecución** selecciona la versión **2026.07**, que usa Python 3.12. Si no tienes Java 17, ejecuta en una celda de Colab `!apt-get -qq update && apt-get -qq install -y openjdk-17-jre-headless` y, en otra celda, `import os; os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"`.

Si faltan las bibliotecas, ejecuta en una celda `%pip install "pyspark>=3.5,<3.6" "numpy<2"` y reinicia el kernel si te lo pide. Puedes consultar la [guía del curso](../01_environment_pyspark.md). Ejecuta las celdas de arriba abajo. Los datos de ejemplo están dentro del notebook y el cálculo se hace en el equipo donde corre el kernel.

In [ ]:
from pyspark.sql import SparkSession, functions as F
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.functions import vector_to_array

spark = (SparkSession.builder.master('local[2]').appName('Regresion logistica')
         .config('spark.sql.shuffle.partitions', '4').getOrCreate())
print(f'Spark {spark.version}')

## 1 · Crear y revisar la tabla de ejemplo

Una fila representa a una persona ficticia. Las horas de formación previa y la jornada semanal se conocen antes de intentar predecir un resultado. El salario mensual y la finalización del curso son dos resultados diferentes. Usamos una semilla para generar la misma tabla en los cuadernos 11, 12 y 14.

La encuesta es **simulada**. Sirve para practicar el método y sus métricas.

In [ ]:
from random import Random

azar = Random(42)
filas = []

for persona in range(1, 361):
    # Tres perfiles de partida hacen visible la idea de segmentación.
    experiencia_base, jornada_base, formacion_base = azar.choice([
        (3, 24, 12), (8, 35, 28), (18, 40, 8),
    ])
    experiencia = max(0, experiencia_base + azar.randint(-3, 3))
    jornada = jornada_base + azar.randint(-4, 4)
    formacion = max(0, formacion_base + azar.randint(-8, 8))
    salario = round(650 + 60 * experiencia + 23 * jornada + azar.gauss(0, 220), 2)
    # La formación previa ayuda y una jornada más larga deja menos tiempo.
    puntuacion = formacion - 0.45 * jornada + azar.gauss(0, 9)
    completa = int(puntuacion >= 1)
    filas.append((persona, experiencia, jornada, formacion, salario, completa))

datos = spark.createDataFrame(filas, [
    'persona', 'anios_experiencia', 'horas_jornada', 'horas_formacion',
    'salario_mensual', 'completa_curso',
])

print(f'Filas: {datos.count()}')
datos.show(5)

## 2 · Preparar las variables y la etiqueta

La etiqueta `completa_curso` vale 1 si la persona termina y 0 si no termina. Usamos solo la formación previa y las horas de jornada. Son datos disponibles antes del resultado. Contamos ambas clases para detectar si una predomina.

In [ ]:
datos.groupBy('completa_curso').count().orderBy('completa_curso').show()
entradas = ['horas_formacion', 'horas_jornada']
ensamblador = VectorAssembler(inputCols=entradas, outputCol='variables')
preparados = ensamblador.transform(datos)

## 3 · Separar entrenamiento y prueba

El modelo aprende solo con **entrenamiento**. Reservamos **prueba** para simular lo que ocurre cuando llegan filas nuevas. El reparto es aproximadamente 80/20. La semilla permite repetir la división con los mismos datos.

In [ ]:
entrenamiento, prueba = preparados.randomSplit([0.8, 0.2], seed=42)
print(f'Entrenamiento: {entrenamiento.count()} · Prueba: {prueba.count()}')

## 4 · Obtener probabilidades y decisiones

El modelo estima una probabilidad para cada clase. Usamos el umbral 0,5 para convertir la probabilidad de terminar en una predicción de 0 o 1. Comparamos esa predicción con la etiqueta real en la parte de prueba.

In [ ]:
modelo = LogisticRegression(
    featuresCol='variables', labelCol='completa_curso', predictionCol='prediccion',
    threshold=0.5,
).fit(entrenamiento)
predicciones = modelo.transform(prueba)
predicciones.select(
    'horas_formacion', 'horas_jornada', 'completa_curso',
    F.round(vector_to_array('probability')[1], 3).alias('probabilidad_completar'),
    'prediccion',
).show(8)

## 5 · Revisar aciertos y errores

La matriz de confusión cruza el resultado real con el predicho. Un falso positivo predice que alguien terminará cuando no lo hace. Un falso negativo predice que no terminará cuando sí lo hace. La exactitud mide qué fracción de la prueba se clasificó correctamente.

In [ ]:
(predicciones.groupBy('completa_curso').pivot('prediccion', [0.0, 1.0])
 .count().fillna(0).orderBy('completa_curso').show())

exactitud = predicciones.agg(
    F.avg((F.col('completa_curso') == F.col('prediccion')).cast('double'))
).first()[0]
mayoria = (entrenamiento.groupBy('completa_curso').count()
           .orderBy(F.desc('count'), 'completa_curso').first()['completa_curso'])
exactitud_referencia = prueba.agg(
    F.avg((F.col('completa_curso') == F.lit(mayoria)).cast('double'))
).first()[0]

print(f'Exactitud del modelo: {exactitud:.1%}')
print(f'Exactitud al predecir siempre la clase {mayoria}: {exactitud_referencia:.1%}')

**Interpretación.** Lee los dos tipos de error además de la exactitud. Si el objetivo fuera ofrecer apoyo a quien corre riesgo de no terminar, habría que decidir qué error tiene más coste antes de elegir un umbral.

## Cerrar la sesión

Liberamos los recursos de Spark al terminar este cuaderno.

In [ ]:
spark.stop()

## Navegación

[Volver al README](../README.md) · [Siguiente: Análisis de texto](13_spark_text_analysis.ipynb) · [Ir a segmentación con K-Means](14_census_kmeans.ipynb)

Documentación de Spark ML. [Regresión y clasificación](https://spark.apache.org/docs/3.5.6/ml-classification-regression.html) · [Preparación de variables](https://spark.apache.org/docs/3.5.6/ml-features.html)